# 04 — Data Cleaning & Preprocessing
**Objective:** Turn raw tables into clean, analysis-ready datasets at separate levels
(order, item, customer, product, seller, geography).
**Principles:** flag instead of delete; document every decision; no target leakage.
**Deliverable:** Cleaned datasets saved to `data/processed/`.

In [1]:
# Cell 2: Load
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "customers":   "customers.csv",
    "geolocation": "geolocation.csv",
    "order_items": "order_items.csv",
    "payments":    "order_payments.csv",
    "reviews":     "order_review.csv",
    "orders":      "orders.csv",
    "products":    "products.csv",
    "sellers":     "sellers.csv",
    "translation": "product_category_name.csv",
}
missing = [f for f in files.values() if not (RAW_DIR / f).exists()]
assert not missing, f"Missing files: {missing}"
data = {name: pd.read_csv(RAW_DIR / f) for name, f in files.items()}
print({k: v.shape for k, v in data.items()})

{'customers': (99441, 5), 'geolocation': (1000163, 5), 'order_items': (112650, 7), 'payments': (103886, 5), 'reviews': (99224, 7), 'orders': (99441, 8), 'products': (32951, 9), 'sellers': (3095, 4), 'translation': (71, 2)}


In [2]:
# Cell 3: orders, flags and derived features

orders = data["orders"].copy()
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

# --- quality flags (rows are kept) ---
orders["flag_delivered_no_date"]       = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()
orders["flag_carrier_before_approval"] = orders["order_delivered_carrier_date"] < orders["order_approved_at"]
orders["flag_carrier_before_purchase"] = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
orders["flag_delivered_before_carrier"] = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

# population for delivery / review analysis
orders["in_delivery_analysis"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].notna()

# --- derived delivery features (in days) ---
DAY = 86400
orders["delivery_days"]  = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / DAY
orders["estimated_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / DAY
orders["delay_days"]     = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.total_seconds() / DAY
orders["is_late"] = (orders["delay_days"] > 0).astype("Int64").where(orders["in_delivery_analysis"])
orders["flag_extreme_delivery"] = orders["delivery_days"] > 60

# --- time features ---
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
orders["purchase_year"]  = orders["order_purchase_timestamp"].dt.year
orders["purchase_dow"]   = orders["order_purchase_timestamp"].dt.dayofweek
orders["purchase_hour"]  = orders["order_purchase_timestamp"].dt.hour
orders["in_trend_window"] = orders["purchase_month"].between(pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-01"))

print("Delivery analysis orders:", orders["in_delivery_analysis"].sum())
print(f"Late rate: {orders['is_late'].mean():.1%}")
print("Orders in trend window:", orders["in_trend_window"].sum())

Delivery analysis orders: 96470
Late rate: 8.1%
Orders in trend window: 99092


C:\Users\ASUS\AppData\Local\Temp\ipykernel_11868\1348180762.py:29: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  orders["purchase_dow"]   = orders["order_purchase_timestamp"].dt.dayofweek


In [3]:
# Cell 4: payments, items and reviews at order level

# --- payments ---
pay = data["payments"].copy()
pay["payment_installments"] = pay["payment_installments"].clip(lower=1)   # 0 installments is impossible

pay_order = pay.groupby("order_id").agg(
    revenue=("payment_value", "sum"),
    n_payments=("payment_sequential", "count"),
    max_installments=("payment_installments", "max"),
).reset_index()

main_type = (pay.sort_values(["order_id", "payment_value"], ascending=[True, False])
                .drop_duplicates("order_id")[["order_id", "payment_type"]]
                .rename(columns={"payment_type": "main_payment_type"}))
pay_order = pay_order.merge(main_type, on="order_id", how="left")

# --- items ---
items = data["order_items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

item_order = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_products=("product_id", "nunique"),
    n_sellers=("seller_id", "nunique"),
    items_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
).reset_index()
item_order["items_total"] = item_order["items_price"] + item_order["freight_value"]

# --- reviews: keep the latest review per order ---
rev = data["reviews"].copy()
rev["review_creation_date"] = pd.to_datetime(rev["review_creation_date"])
rev["review_answer_timestamp"] = pd.to_datetime(rev["review_answer_timestamp"])
rev = rev.sort_values(["order_id", "review_answer_timestamp"]).drop_duplicates("order_id", keep="last")
rev["has_comment"] = rev["review_comment_message"].notna()
reviews_clean = rev[["order_id", "review_id", "review_score", "has_comment",
                     "review_creation_date", "review_answer_timestamp"]].reset_index(drop=True)

print("pay_order:", pay_order.shape, "| item_order:", item_order.shape, "| reviews_clean:", reviews_clean.shape)

pay_order: (99440, 5) | item_order: (98666, 7) | reviews_clean: (98673, 6)


In [4]:
# Cell 5: geography, customers and sellers

geo = data["geolocation"].copy()
in_brazil = geo["geolocation_lat"].between(-34, 5.3) & geo["geolocation_lng"].between(-74, -34)
print("Geolocation rows removed (outside Brazil):", (~in_brazil).sum())
geo = geo[in_brazil]

geo_zip = (geo.groupby("geolocation_zip_code_prefix")
              .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
              .reset_index()
              .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"}))
print("geo_zip:", geo_zip.shape, "| unique zip:", geo_zip["zip_code_prefix"].is_unique)

customers_clean = (data["customers"]
    .merge(geo_zip, left_on="customer_zip_code_prefix", right_on="zip_code_prefix", how="left")
    .drop(columns="zip_code_prefix")
    .rename(columns={"lat": "customer_lat", "lng": "customer_lng"}))

sellers_clean = (data["sellers"]
    .merge(geo_zip, left_on="seller_zip_code_prefix", right_on="zip_code_prefix", how="left")
    .drop(columns="zip_code_prefix")
    .rename(columns={"lat": "seller_lat", "lng": "seller_lng"}))

assert len(customers_clean) == 99441 and len(sellers_clean) == 3095
print("Customers without coordinates:", customers_clean["customer_lat"].isna().sum())
print("Sellers without coordinates:  ", sellers_clean["seller_lat"].isna().sum())

Geolocation rows removed (outside Brazil): 42
geo_zip: (19010, 3) | unique zip: True
Customers without coordinates: 279
Sellers without coordinates:   7


In [5]:
# Cell 6: products

prod = data["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

trans = pd.concat([data["translation"], pd.DataFrame({
    "product_category_name": ["portateis_cozinha_e_preparadores_de_alimentos", "pc_gamer"],
    "product_category_name_english": ["portable_kitchen_food_preparers", "pc_gamer"],
})], ignore_index=True)

products_clean = prod.merge(trans, on="product_category_name", how="left")
assert len(products_clean) == len(prod)

products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")
products_clean["product_category_name_english"] = products_clean["product_category_name_english"].fillna("unknown")
products_clean.loc[products_clean["product_weight_g"] == 0, "product_weight_g"] = np.nan
products_clean["product_volume_cm3"] = (products_clean["product_length_cm"] *
                                        products_clean["product_height_cm"] *
                                        products_clean["product_width_cm"])

print("Unknown category:", (products_clean["product_category_name_english"] == "unknown").sum())
print("Missing weight:  ", products_clean["product_weight_g"].isna().sum())

Unknown category: 610
Missing weight:   6


In [6]:
# Cell 7: build the order-level and item-level datasets

orders_clean = (orders
    .merge(customers_clean[["customer_id", "customer_unique_id", "customer_city",
                            "customer_state", "customer_zip_code_prefix"]], on="customer_id", how="left")
    .merge(pay_order, on="order_id", how="left")
    .merge(item_order, on="order_id", how="left")
    .merge(reviews_clean[["order_id", "review_score", "has_comment"]], on="order_id", how="left"))

orders_clean["has_items"]   = orders_clean["n_items"].notna()
orders_clean["has_payment"] = orders_clean["revenue"].notna()
# a "valid sale" excludes canceled/unavailable orders and orders without payment info
orders_clean["is_valid_sale"] = (~orders_clean["order_status"].isin(["canceled", "unavailable"])) & orders_clean["has_payment"]

items_clean = (items
    .merge(products_clean[["product_id", "product_category_name_english", "product_weight_g", "product_volume_cm3"]],
           on="product_id", how="left")
    .merge(sellers_clean[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left")
    .merge(orders_clean[["order_id", "customer_unique_id", "customer_state", "order_status",
                         "order_purchase_timestamp", "order_estimated_delivery_date",
                         "order_delivered_customer_date", "delivery_days", "delay_days",
                         "is_late", "in_delivery_analysis", "purchase_month"]],
           on="order_id", how="left"))
items_clean["item_total"] = items_clean["price"] + items_clean["freight_value"]

print("orders_clean:", orders_clean.shape, "| items_clean:", items_clean.shape)

orders_clean: (99441, 42) | items_clean: (112650, 24)


In [7]:
# Cell 8: validation checks

assert orders_clean["order_id"].is_unique and len(orders_clean) == 99441
assert len(items_clean) == 112650
assert reviews_clean["order_id"].is_unique
assert len(products_clean) == 32951
assert orders_clean["customer_unique_id"].notna().all()

print("Orders with revenue:            ", orders_clean["has_payment"].sum())
print("Valid sales:                    ", orders_clean["is_valid_sale"].sum())
print(f"Revenue, all orders:             R$ {orders_clean['revenue'].sum():,.2f}")
print(f"Revenue, valid sales only:       R$ {orders_clean.loc[orders_clean['is_valid_sale'], 'revenue'].sum():,.2f}")
print("Delivery-analysis orders:       ", orders_clean["in_delivery_analysis"].sum())
print(f"Late rate:                       {orders_clean['is_late'].mean():.1%}")
print(f"Average review (delivered):      {orders_clean.loc[orders_clean['in_delivery_analysis'], 'review_score'].mean():.2f}")
print("\nMissing values in key order columns:")
display(orders_clean[["revenue", "n_items", "review_score", "delivery_days", "is_late"]].isna().sum().to_frame("missing"))

Orders with revenue:             99440
Valid sales:                     98206
Revenue, all orders:             R$ 16,008,872.12
Revenue, valid sales only:       R$ 15,739,137.01
Delivery-analysis orders:        96470
Late rate:                       8.1%
Average review (delivered):      4.16

Missing values in key order columns:


,missing
revenue,1
n_items,775
review_score,768
delivery_days,2965
is_late,2971


In [8]:
# Cell 9: Save

outputs = {
    "orders_clean": orders_clean,
    "items_clean": items_clean,
    "customers_clean": customers_clean,
    "sellers_clean": sellers_clean,
    "products_clean": products_clean,
    "reviews_clean": reviews_clean,
    "geo_zip": geo_zip,
}
for name, df in outputs.items():
    df.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    print(f"Saved {name:16s} {df.shape}")

Saved orders_clean     (99441, 42)
Saved items_clean      (112650, 24)
Saved customers_clean  (99441, 7)
Saved sellers_clean    (3095, 6)
Saved products_clean   (32951, 11)
Saved reviews_clean    (98673, 6)
Saved geo_zip          (19010, 3)


## Cleaning log
| Step | Decision | Reason |
|------|----------|--------|
| Dates | Converted all timestamps to datetime | Needed for durations |
| Delivery population | `in_delivery_analysis` = delivered and has delivery date (96,470) | 8 delivered orders have no date |
| Timestamp anomalies | Flagged, not removed; carrier/approval times never used as features | Logging issues; avoids leakage |
| Extreme deliveries | Kept, flagged `flag_extreme_delivery` | Genuine logistics failures |
| Revenue | `revenue` = sum of payment_value per order; `is_valid_sale` excludes canceled/unavailable | Matches items+freight within 0.02% |
| Installments | 0 clipped to 1 | Impossible value |
| Reviews | Latest review per order | 547 orders had multiple |
| Geolocation | Removed 42 points outside Brazil; median lat/lng per zip | 26% duplicates would multiply rows |
| Products | Missing category set to "unknown"; 2 manual translations; weight 0 set to missing | Data errors |
| Structure | Separate datasets by level (order, item, customer, product, seller) | Avoids one huge table |
| Trend window | `in_trend_window` = Jan 2017 to Aug 2018 | 2016 and Sep-Oct 2018 are incomplete |

## Deliverable
Cleaned datasets saved in `data/processed/`. Next: **05 — Exploratory Data Analysis**.